# 1.1 — Carga y visualización inicial

Carga imágenes con PIL y con OpenCV para cada dataset.
Verifica que `pad_image_to_multiple()` de `src/utils/images.py` funciona
correctamente sobre imágenes de cada dataset y visualiza el resultado del padding.

**Autor:** Jorge Ceferino Valdez — Maestría en Informática y Sistemas (UNPA)

In [1]:
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import pandas as pd
import cv2
from PIL import Image

In [2]:
project_root = Path(os.path.abspath('')).parent
if project_root not in sys.path:
    sys.path.insert(0, str(project_root))
print(f'project_root: {project_root}')

project_root: /home/jorge/development/drone-waste-detect-compress


In [3]:
from src.config import config, external_datasets, project_dir, interim_data_dir
from src.utils.images import pad_image_to_multiple

In [4]:
# Directorio unificado con todas las imágenes convertidas a PNG (generado en notebook 1.0)
image_path = interim_data_dir() / 'dataset_png'
print(f'Cargando imágenes desde: {image_path}')

Cargando imágenes desde: /home/jorge/development/drone-waste-detect-compress/data/interim/dataset_png


## 1. Recolección de imágenes por clase

In [5]:
# Recolectar imágenes organizadas por subdirectorio (= clase o dataset de origen)
imagenes_por_clase = defaultdict(list)
for subdir, _, files in os.walk(image_path):
    for file in files:
        if file.endswith('.png'):
            clase = os.path.basename(subdir)
            imagenes_por_clase[clase].append(Path(subdir) / file)

total_imagenes = sum(len(v) for v in imagenes_por_clase.values())
print(f'Total de imágenes: {total_imagenes}')
print(f'Clases / datasets encontrados: {list(imagenes_por_clase.keys())}')

Total de imágenes: 0
Clases / datasets encontrados: []


## 2. Carga con PIL

In [ ]:
# Carga de ejemplo con PIL para la primera imagen disponible
for clase, imagenes in list(imagenes_por_clase.items())[:3]:
    img_path = imagenes[0]
    pil_img = Image.open(img_path).convert('RGB')
    print(f'[PIL] {clase}/{img_path.name}: modo={pil_img.mode}, tamaño={pil_img.size}')

## 3. Carga con OpenCV

In [ ]:
# Carga de ejemplo con OpenCV (BGR → RGB para visualizar)
for clase, imagenes in list(imagenes_por_clase.items())[:3]:
    img_path = imagenes[0]
    cv_img = cv2.imread(str(img_path))
    h, w = cv_img.shape[:2]
    print(f'[CV2] {clase}/{img_path.name}: shape={cv_img.shape}, dtype={cv_img.dtype}')

## 4. Grid de imágenes por clase/dataset

In [ ]:
random.seed(42)
# Una imagen por clase en grilla
clases = list(imagenes_por_clase.keys())
n = len(clases)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flat if n > 1 else [axes]

for ax, clase in zip(axes, clases):
    img_path = random.choice(imagenes_por_clase[clase])
    img = cv2.imread(str(img_path))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(clase, fontsize=10, fontweight='bold')
    ax.axis('off')

for ax in list(axes)[n:]:
    ax.axis('off')

plt.suptitle('Una imagen aleatoria por clase/dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Verificación de `pad_image_to_multiple()`

Verifica que el padding a múltiplo de 64 (requerimiento de CompressAI)
funciona correctamente sobre imágenes de cada dataset.

In [ ]:
pad_multiple = config['compressai']['training']['pad_multiple']  # 64
print(f'pad_multiple: {pad_multiple}')

random.seed(42)
for clase, imagenes in list(imagenes_por_clase.items())[:4]:
    img_path = random.choice(imagenes)
    pil_img = Image.open(img_path).convert('RGB')
    padded  = pad_image_to_multiple(pil_img, multiple=pad_multiple)

    orig_w, orig_h = pil_img.size
    pad_w,  pad_h  = padded.size

    assert pad_w % pad_multiple == 0, f'Ancho {pad_w} no es múltiplo de {pad_multiple}'
    assert pad_h % pad_multiple == 0, f'Alto  {pad_h} no es múltiplo de {pad_multiple}'

    print(f'  {clase}: {orig_w}x{orig_h} → {pad_w}x{pad_h} (múltiplo de {pad_multiple})')

print('pad_image_to_multiple() verificado correctamente.')

In [ ]:
# Visualización de padding: original vs padded
random.seed(0)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f'pad_image_to_multiple(multiple={pad_multiple}) — original vs padded', fontsize=12)

clases_sample = list(imagenes_por_clase.keys())[:4]
for col, clase in enumerate(clases_sample):
    img_path = random.choice(imagenes_por_clase[clase])
    pil_img  = Image.open(img_path).convert('RGB')
    padded   = pad_image_to_multiple(pil_img, multiple=pad_multiple)

    axes[0, col].imshow(pil_img)
    axes[0, col].set_title(f'Original\n{pil_img.size[0]}x{pil_img.size[1]}', fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(padded)
    axes[1, col].set_title(f'Padded\n{padded.size[0]}x{padded.size[1]}', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

## 6. Análisis de calidad de imagen

Verifica imágenes borrosas, sobreexpuestas y subexpuestas.

In [ ]:
def es_borrosa(img_gray, umbral=100):
    return cv2.Laplacian(img_gray, cv2.CV_64F).var() < umbral

def es_sobreexpuesta(img_gray, umbral=230):
    return cv2.mean(img_gray)[0] > umbral

def es_subexpuesta(img_gray, umbral=25):
    return cv2.mean(img_gray)[0] < umbral

dimensiones    = defaultdict(int)
imagenes_borrosas       = []
imagenes_sobreexpuestas = []
imagenes_subexpuestas   = []
data = []

for clase, imagenes in imagenes_por_clase.items():
    for img_path in imagenes:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        h, w = img.shape
        dimensiones[(w, h)] += 1
        data.append([img_path.name, clase, w, h])
        if es_borrosa(img):        imagenes_borrosas.append(img_path)
        if es_sobreexpuesta(img):  imagenes_sobreexpuestas.append(img_path)
        if es_subexpuesta(img):    imagenes_subexpuestas.append(img_path)

print(f'Imágenes analizadas:      {len(data)}')
print(f'  Borrosas:               {len(imagenes_borrosas)}')
print(f'  Sobreexpuestas:         {len(imagenes_sobreexpuestas)}')
print(f'  Subexpuestas:           {len(imagenes_subexpuestas)}')

In [ ]:
# Distribución de resoluciones
df = pd.DataFrame(data, columns=['Imagen', 'Clase', 'Ancho', 'Alto'])
grouped = df.groupby(['Ancho', 'Alto']).size().reset_index(name='Cantidad').sort_values('Cantidad', ascending=False)
print(grouped.head(15).to_string(index=False))

## 7. Verificación final

In [ ]:
print('=' * 55)
print('VERIFICACIÓN FINAL')
print('=' * 55)
assert image_path.exists(), f'No existe: {image_path}'
assert total_imagenes > 0, 'No se encontraron imágenes'
print(f'  [OK] dataset_png: {total_imagenes} imágenes')
print(f'  [OK] pad_image_to_multiple verificado')
print()
print('Notebook 1.1 completado correctamente.')